# C1-ml-fundamentals — Practice p18 — Solution

*Coding is required.*

Under class imbalance a single accuracy number misleads, so reports should
carry their own context. Write code to implement

```python
def imbalance_report(y_true, y_pred):
    ...
```

returning the tuple `(accuracy, baseline_accuracy, recall, f1)` as four
floats, where

- `baseline_accuracy` is the accuracy the always-majority constant rule
  would score on `y_true` (the larger of the two class fractions);
- `f1` uses the convention from the lesson: if the rule makes no positive
  calls or finds no true positives, `f1` is `0.0` (an `if` statement is
  fine — only loops are banned).

**Banned inside the function: Python `for` and `while` loops and list
comprehensions. Any use of a banned construct scores zero points for this
problem.**

Apply it to the factory data below for both detectors and print the two
reports. One detector beats the baseline in the only metric that matters —
say which, in a one-line comment.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
y_true = np.zeros(100, dtype=int)
y_true[:6] = 1                      # 6 defective parts

pred_detector = np.zeros(100, dtype=int)
pred_detector[:4] = 1               # catches 4 of the 6 defects
pred_detector[6:9] = 1              # 3 false alarms

pred_silent = np.zeros(100, dtype=int)   # never flags anything

In [ ]:
def imbalance_report(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    n = y_true.size

    accuracy = (TP + TN) / n
    pos_frac = np.mean(y_true == 1)
    baseline_accuracy = max(pos_frac, 1 - pos_frac)
    recall = TP / (TP + FN)
    if TP == 0:                       # no positives found (or none called)
        f1 = 0.0
    else:
        precision = TP / (TP + FP)
        f1 = 2 * precision * recall / (precision + recall)
    return float(accuracy), float(baseline_accuracy), float(recall), float(f1)

print("detector:", imbalance_report(y_true, pred_detector))
print("silent:  ", imbalance_report(y_true, pred_silent))
# The detector wins on recall (0.667 vs 0.0): it actually finds defects.

**Explanation.** The baseline is computed from `y_true` alone: a constant
rule scores the majority-class fraction, here 0.94, and any accuracy must be
read against it. The detector's 0.95 accuracy barely clears that bar, but its
recall 2/3 and F1 ≈ 0.62 show real detection ability, while the silent rule's
0.94 accuracy comes with recall 0 — the report format makes the imbalance
trap visible by construction. The `TP == 0` guard implements the lesson's
convention that a rule finding no positives earns F1 = 0 rather than a
division error.

### Answer check

In [ ]:
acc, base, rec, f1 = imbalance_report(y_true, pred_detector)
assert np.isclose(acc, 0.95, atol=1e-9, rtol=0) and np.isclose(base, 0.94, atol=1e-9, rtol=0)
assert np.isclose(rec, 0.6666666666666666, atol=1e-9, rtol=0) and np.isclose(f1, 0.6153846153846153, atol=1e-9, rtol=0)
acc_s, base_s, rec_s, f1_s = imbalance_report(y_true, pred_silent)
assert np.isclose(acc_s, 0.94, atol=1e-9, rtol=0) and np.isclose(base_s, 0.94, atol=1e-9, rtol=0)
assert rec_s == 0.0 and f1_s == 0.0
import inspect
src = inspect.getsource(imbalance_report)
assert "for " not in src and "while " not in src
print("p18 OK")